In [7]:
!pip install playwright pandas
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 1.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]━━━━━━━ 2/3 [playwright]
(node:47287) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
162.3 MiB [                    ] 0% 0.0s162.3 MiB [                    ] 0% 453.5s162.3 MiB [                    ] 0% 807.0s162.3 MiB [                    ] 0% 1070.4s162.3 MiB [                    ] 0% 943.3s162.3 MiB [                    ] 0% 1211.1s162.3 MiB [                    ] 0% 1054.3s162.3 MiB [                    ] 0% 963.0s162.3 MiB [                    ] 0% 899.7s162.3 MiB [                    ] 0% 860.8s162.3 MiB [                    ] 0% 846.3s162.3 MiB [                    ] 0% 793.8s162.3 MiB [

In [9]:
from playwright.async_api import async_playwright
import asyncio
import json
from datetime import datetime, timedelta

# --- CONFIGURATION ---
X_AUTH_TOKEN = "redacted" # Put your token back

# We removed 'f=live' and added placeholders for dates
BASE_QUERY = "from:PIBFactCheck (bank OR RBI OR SBI OR finance OR loan OR tax OR fraud OR scam OR rupees OR UPI OR investment OR mutual) filter:media"

# Start from Jan 2023 up to today
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2026, 4, 8)
CHUNK_DAYS = 30 # Search one month at a time

def get_date_ranges():
    ranges = []
    current = START_DATE
    while current < END_DATE:
        next_date = current + timedelta(days=CHUNK_DAYS)
        if next_date > END_DATE:
            next_date = END_DATE
        ranges.append((current.strftime("%Y-%m-%d"), next_date.strftime("%Y-%m-%d")))
        current = next_date
    return ranges

async def run_scraper():
    scraped_data = []
    seen_tweets = set()
    date_ranges = get_date_ranges()
    
    print(f"Generated {len(date_ranges)} timeboxed searches to execute...")

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled"]
        )
        
        context = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        
        await context.add_cookies([{"name": "auth_token", "value": X_AUTH_TOKEN, "domain": ".x.com", "path": "/"}])
        page = await context.new_page()

        for since_date, until_date in date_ranges:
            # Build the specific URL for this month
            url_query = f"{BASE_QUERY} since:{since_date} until:{until_date}"
            search_url = f"https://x.com/search?q={url_query}&src=typed_query"
            
            print(f"\n🗓️ Searching: {since_date} to {until_date}...")
            await page.goto(search_url)
            
            try:
                # Give it a max of 8 seconds to find a tweet. If not, this month is empty, skip it.
                await page.wait_for_selector('[data-testid="tweet"]', timeout=8000)
                await asyncio.sleep(2) 
            except Exception:
                print("  -> No tweets found for this month. Moving on.")
                continue

            # Just grab what's immediately on the screen and maybe scroll ONCE. 
            # We don't need to dig deep because the timeframe is so small.
            for _ in range(2): 
                tweets = await page.locator('[data-testid="tweet"]').all()
                for tweet in tweets:
                    try:
                        text_loc = tweet.locator('[data-testid="tweetText"]')
                        tweet_text = await text_loc.inner_text() if await text_loc.count() > 0 else ""
                        
                        photo_loc = tweet.locator('[data-testid="tweetPhoto"] img')
                        if await photo_loc.count() > 0:
                            image_url = await photo_loc.first.get_attribute('src')
                            
                            if image_url and "media" in image_url and image_url not in seen_tweets:
                                seen_tweets.add(image_url)
                                scraped_data.append({
                                    "tweet_text": tweet_text,
                                    "image_url": image_url
                                })
                                print(f"  ✅ [Total: {len(seen_tweets)}] Extracted: {tweet_text[:30].replace(chr(10), ' ')}...")
                    except Exception:
                        continue
                
                await page.mouse.wheel(0, 2000)
                await asyncio.sleep(1.5)

        await context.close()
        await browser.close()
        
    return scraped_data

data = await run_scraper()

if data:
    with open("raw_tweets_bulk.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"\n🎉 BOOM. Extracted {len(data)} images. Saved to raw_tweets_bulk.json")
else:
    print("\n❌ No data extracted.")

Generated 40 timeboxed searches to execute...

🗓️ Searching: 2023-01-01 to 2023-01-31...
  ✅ [Total: 1] Extracted: An approval letter claims to g...
  ✅ [Total: 2] Extracted: The #Youtube channel claims th...
  ✅ [Total: 3] Extracted: In another video, Nation Tv ha...
  ✅ [Total: 4] Extracted: Does writing anything on the b...
  ✅ [Total: 5] Extracted: #FoundTrue   Yes! BHIM UPI App...
  ✅ [Total: 6] Extracted: "Chance to win a Fuel Subsidy ...

🗓️ Searching: 2023-01-31 to 2023-03-02...
  ✅ [Total: 7] Extracted: An approval letter claims to g...
  ✅ [Total: 8] Extracted: A #FAKE lucky draw in the name...
  ✅ [Total: 9] Extracted: 𝐂𝐥𝐚𝐢𝐦: Customers need to updat...
  ✅ [Total: 10] Extracted: It is being claimed in a notic...
  ✅ [Total: 11] Extracted: Does writing anything on the b...
  ✅ [Total: 12] Extracted: A tweet claims that the centra...

🗓️ Searching: 2023-03-02 to 2023-04-01...
  ✅ [Total: 13] Extracted: An order issued in the name of...
  ✅ [Total: 14] Extracted: A #Fake collat

In [10]:
import json
import os
import requests
import pandas as pd
from io import BytesIO
from PIL import Image
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# --- CONFIGURATION ---
GEMINI_API_KEY = "redacted" # <-- Paste your API key here
LOCAL_JSON_FILE = "raw_tweets_bulk.json"    # <-- Pointing to your new 112-tweet dataset!

# Initialize Gemini client
ai_client = genai.Client(api_key=GEMINI_API_KEY)

# --- NLP SCHEMA DEFINITION ---
# This forces the LLM to return data exactly in this structure
class MisinfoData(BaseModel):
    target_institution: str = Field(description="The bank or govt body being impersonated (e.g., SBI, RBI, Income Tax). If none, write 'Unknown'.")
    scam_vector: str = Field(description="How the scam is delivered: 'WhatsApp', 'SMS', 'Website', 'News Article', or 'Other'.")
    misinfo_type: str = Field(description="Categorize the scam: e.g., 'Account Block', 'Fake Scheme', 'Loan Fraud'.")
    core_claim: str = Field(description="A concise 1-sentence summary of the false claim.")
    extracted_text: str = Field(description="The readable text from the scam image, ignoring the red 'FAKE' watermarks.")

# --- PHASE 2 & 3: VISION & NLP PIPELINE ---
def process_tweet_content(tweet_text, image_url):
    try:
        # 1. Download the image into memory
        response = requests.get(image_url)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        
        # 2. Build the prompt
        prompt = f"""
        Analyze the attached image and the official tweet text from PIB Fact Check.
        The image contains financial misinformation with a 'FAKE' watermark over it.
        Read through the watermark and extract the details of the scam.
        
        Official Tweet Text: "{tweet_text}"
        """
        
        # 3. Call Gemini Multimodal with Structured Outputs
        result = ai_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[img, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=MisinfoData,
                temperature=0.1 # Low temp to prevent hallucinations
            ),
        )
        return result.text
        
    except Exception as e:
        print(f"  -> Error processing image {image_url}: {e}")
        return None

# --- PHASE 4: EXECUTION & CSV EXPORT ---
def main():
    if not os.path.exists(LOCAL_JSON_FILE):
        print(f"❌ Error: Could not find {LOCAL_JSON_FILE}. Make sure it's in the same folder!")
        return

    # Load the data your bot just scraped
    print(f"Loading raw tweets from {LOCAL_JSON_FILE}...")
    with open(LOCAL_JSON_FILE, 'r', encoding='utf-8') as f:
        raw_tweets = json.load(f)
    
    if not raw_tweets:
        print("No tweets found in the JSON file. Exiting.")
        return

    processed_data = []
    
    print(f"\nProcessing {len(raw_tweets)} images and structuring data with Gemini...")
    for i, tweet in enumerate(raw_tweets):
        print(f"Processing image {i+1}/{len(raw_tweets)}...")
        
        # Pass the data to the LLM
        structured_json_str = process_tweet_content(tweet["tweet_text"], tweet["image_url"])
        
        if structured_json_str:
            try:
                extracted_info = json.loads(structured_json_str)
                
                # Merge original tweet data with extracted LLM data
                row = {
                    "Institution": extracted_info.get("target_institution", "Unknown"),
                    "Scam_Vector": extracted_info.get("scam_vector", "Unknown"),
                    "Misinfo_Type": extracted_info.get("misinfo_type", "Unknown"),
                    "Core_Claim": extracted_info.get("core_claim", ""),
                    "Original_Tweet": tweet["tweet_text"].replace('\n', ' '),
                    "Image_Text": extracted_info.get("extracted_text", "").replace('\n', ' '),
                    "Image_URL": tweet["image_url"]
                }
                processed_data.append(row)
            except json.JSONDecodeError:
                print(f"  -> Failed to parse JSON from Gemini for an image.")

    # Compile to CSV
    if processed_data:
        df = pd.DataFrame(processed_data)
        output_file = "pib_financial_factchecks.csv"
        df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"\n✅ Success! Structured data saved to {output_file}")
        
        # Display the first few rows
        print("\nPreview of extracted data:")
        from IPython.display import display
        display(df[['Institution', 'Misinfo_Type', 'Core_Claim']].head())
    else:
        print("\n❌ No data successfully processed.")

if __name__ == "__main__":
    main()

Loading raw tweets from raw_tweets_bulk.json...

Processing 112 images and structuring data with Gemini...
Processing image 1/112...
Processing image 2/112...
Processing image 3/112...
Processing image 4/112...
Processing image 5/112...
Processing image 6/112...
Processing image 7/112...
  -> Failed to parse JSON from Gemini for an image.
Processing image 8/112...
Processing image 9/112...
Processing image 10/112...
Processing image 11/112...
Processing image 12/112...
Processing image 13/112...
Processing image 14/112...
Processing image 15/112...
Processing image 16/112...
Processing image 17/112...
Processing image 18/112...
Processing image 19/112...
Processing image 20/112...
Processing image 21/112...
Processing image 22/112...
Processing image 23/112...
Processing image 24/112...
Processing image 25/112...
Processing image 26/112...
Processing image 27/112...
Processing image 28/112...
Processing image 29/112...
Processing image 30/112...
Processing image 31/112...
Processing im

,Institution,Misinfo_Type,Core_Claim
0,Pradhan Mantri Mudra Yojana,Loan Fraud,An approval letter falsely claims to grant a l...
1,Indian Government,Political Misinformation,Finance Minister Nirmala Sitharaman has resigned.
2,Indian Government,Fake News,The false claim is that Finance Minister Nirma...
3,Reserve Bank of India,Currency Validity,New guidelines from the Reserve Bank of India ...
4,BHIM / NPCI,Misleading Offer,The SMS claims that users can receive an exclu...
